# **RetrievalQA**
We could have the Language Model (LLM) read the paper and summarize it for us, or create a QA bot that can answer our questions based on a given paper.

In [16]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [2]:
# We can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

Initializing a Meta-Llama Model Using IBM Watsonx.ai and LangChain:

In [3]:
parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.2, # this randomness or creativity of the model's responses 
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above and fill in the API key when running locally
}

project_id = "skills-network"
llama_model = ModelInference(
    model_id='meta-llama/llama-4-maverick-17b-128e-instruct-fp8',
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
llama_llm = WatsonxLLM(model=llama_model)

Loading the Doc into a Document Object, and creating chunks:

In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

In [5]:
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks_container = text_splitter.split_documents(document)

Configuring Text Embedding Models with IBM Watsonx and LangChain:

In [6]:
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
embed_params = {
 EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
 EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

from langchain_ibm import WatsonxEmbeddings
watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual", #This is IBM's Embedding Model. OpenAI, Hugging Face, and others offer embedding models too. Here, we will use the embedding model from IBM's watsonx.ai to work with the text.
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params,
)

Creating a Vector Store and Retriever:

In [7]:
from langchain.vectorstores import Chroma
docsearch = Chroma.from_documents(chunks_container, watsonx_embedding)
retriever = docsearch.as_retriever()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [9]:
from langchain.chains import RetrievalQA
# Create a RetrievalQA chain by configuring:
qa = RetrievalQA.from_chain_type(
    # The language model to use for generating answers
    llm=llama_llm,
    
    # The chain type "stuff" means all retrieved documents are simply concatenated and passed to the LLM
    chain_type="stuff",
    
    # The retriever component that will fetch relevant documents
    # docsearch.as_retriever() converts the vector store into a retriever interface
    retriever=docsearch.as_retriever(),
    
    # Whether to include the source documents in the response
    # Set to False to return only the generated answer
    return_source_documents=False
)

# Define a query to test the QA system
# This question asks about the main topic of the paper
query = "what is this paper discussing?"

# Execute the QA chain with the query
# This will:
# 1. Send the query to the retriever to get relevant documents
# 2. Combine those documents using the "stuff" method
# 3. Send the query and combined documents to the Llama LLM
# 4. Return the generated answer (without source documents)
qa.invoke(query)

{'query': 'what is this paper discussing?',
 'result': ' This paper is discussing the MindGuide app, a chatbot that serves as a mental health assistant.'}

What happened here:<br>
The AI just read a PDF and answered a question about it.<br>
And it never saw the whole PDF, it only saw the 3-4 most relevant chunks that Chroma found.<br>
.......................................................<br>
The full pipeline that just ran invisibly:<br>
"what is this paper discussing?"<br>
            ↓
Convert query to vector<br>
            ↓
Chroma finds 4 most relevant chunks from the PDF<br>
            ↓
Those chunks get "stuffed" into a prompt like:<br>
"Answer this question: what is this paper discussing?<br>
 Using this context: [chunk1] [chunk2] [chunk3] [chunk4]"<br>
            ↓<br>
LLM reads that prompt and answers<br>
            ↓<br>
"This paper is discussing the MindGuide app..."<br>
.......................................................<br>
This is RAG, complete and working:<br>
Retrieval — Chroma found relevant chunks<br>
Augmented — those chunks augmented the prompt<br>
Generation — LLM generated an answer based on them<br>

In [11]:
qa.invoke("What LangChain components does MindGuide use?")

{'query': 'What LangChain components does MindGuide use?',
 'result': '## Step 1\nThe relevant text is in Section III. ARCHITECTURE, subsection A. ChatModel.\n\n## Step 2\nThe text states that "The MindGuide Bot uses below components from LangChain."\n\n## Step 3\nAlthough the specific components are not listed in the given text, it is mentioned that the types of messages supported in LangChain are SystemMessage, HumanMessage, and AIMessage, and that the MindGuide Bot uses components from LangChain.\n\n## Step 4\nSince the text only explicitly discusses "ChatModel" as a component and mentions types of messages, we can infer that at least "ChatModel" is used.\n\nThe best answer is ChatModel.'}

What the LLM actually did:<br>
It showed its reasoning step by step:<br>
Found the relevant section (Architecture → ChatModel)<br>
Quoted the relevant part<br>
Admitted what it didn't know ("specific components not listed")<br>
Made a reasonable inference<br>
Gave a final answer: ChatModel<br>

In [12]:
qa.invoke("What is the purpose of MindGuide?")

{'query': 'What is the purpose of MindGuide?',
 'result': " Unfortunately, the provided context does not explicitly state the purpose of MindGuide. I don't know."}

In [13]:
qa.invoke("What mental health problem does MindGuide solve?")

{'query': 'What mental health problem does MindGuide solve?',
 'result': " I don't know."}

In [15]:
# To check what chunks get retrieved..
# This shows us exactly what context the LLM was given.
# If the chunks are irrelevant, the answer will be bad regardless of how good the LLM is.
docs = docsearch.similarity_search("purpose of MindGuide")
for doc in docs:
    print(doc.page_content[:200])
    print("---")

functionality that sends the user's input (question) as a 
chat prompt template to the LangChain framework. This 
input serves as the "human message prompt" template .
---
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models 
(LLMs) . This framework though the rapid delivery of building
---
motivation. Your ultimate goal is to equip the 
patient with the tools and skills needed to navigate 
life's challenges with confidence and resilience .
---
contextualized language models to introduce MindGuide, an 
innovative chatbot serving as a mental health assistant for 
individuals seeking guidance and support in these critical areas.
---
